# Requirement 3 — Approach 2: End-to-End Deep Learning with EfficientNet-B0
**Course:** CAI3105/CS460 — Deep Learning  
**Dataset:** Brain Tumor MRI (Kaggle)  
**Model:** EfficientNet-B0 (End-to-End Fine-Tuned Classifier)

## Step 1 — Mount Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

zip_path = '/content/drive/MyDrive/Brain_Tumor/archive.zip'
extract_path = '/content/brain_tumor_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")


## Step 2 — Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

print("TensorFlow version:", tf.__version__)


## Step 3 — Data Preprocessing & Augmentation

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
LEARNING_RATE = 0.001
EPOCHS = 15

TRAIN_DIR = '/content/brain_tumor_data/Training'
TEST_DIR  = '/content/brain_tumor_data/Testing'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    fill_mode='nearest',
    validation_split=0.2   # 80% train, 20% validation
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_generator.class_indices.keys())
print("Classes:", CLASS_NAMES)
print("Training samples  :", train_generator.samples)
print("Validation samples:", val_generator.samples)
print("Testing samples   :", test_generator.samples)


## Step 4 — Build End-to-End EfficientNet-B0 Model

In [ ]:
# Load base model (frozen weights from ImageNet)
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze base

# Build end-to-end model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f"\nTotal trainable parameters: {model.count_params():,}")


## Step 5 — Define Callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]


## Step 6 — Train the End-to-End Model

In [ ]:
print("Starting End-to-End training...")
start_time = time.time()

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

total_train_time = time.time() - start_time
print(f"\nTotal training time: {total_train_time:.2f} seconds ({total_train_time/60:.1f} minutes)")


## Step 7 — Plot Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curve
axes[0].plot(history.history['accuracy'],     label='Train Accuracy',     color='steelblue',  linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',       color='darkorange', linewidth=2, linestyle='--')
axes[0].set_title('Accuracy — End-to-End EfficientNet-B0', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curve
axes[1].plot(history.history['loss'],     label='Train Loss',     color='steelblue',  linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',       color='darkorange', linewidth=2, linestyle='--')
axes[1].set_title('Loss — End-to-End EfficientNet-B0', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/learning_curves_approach2.png', dpi=150)
plt.show()
print("Learning curves saved.")


## Step 8 — Evaluate on Test Set

In [ ]:
print("Evaluating on test set...")
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted')
rec  = recall_score(y_true, y_pred, average='weighted')
f1   = f1_score(y_true, y_pred, average='weighted')

print("=" * 52)
print("  Approach-2: End-to-End EfficientNet-B0 Results")
print("=" * 52)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  Precision : {prec*100:.2f}%")
print(f"  Recall    : {rec*100:.2f}%")
print(f"  F1-Score  : {f1*100:.2f}%")
print("=" * 52)

print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


## Step 9 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title('Confusion Matrix — Approach 2: End-to-End EfficientNet-B0', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('/content/confusion_matrix_approach2.png', dpi=150)
plt.show()
print("Confusion matrix saved.")


## Step 10 — Save Results for Comparative Analysis

In [ ]:
import json

results_approach2 = {
    "approach": "End-to-End EfficientNet-B0 (Fine-Tuned)",
    "accuracy":  round(acc * 100, 2),
    "precision": round(prec * 100, 2),
    "recall":    round(rec * 100, 2),
    "f1_score":  round(f1 * 100, 2),
    "total_training_time_sec": round(total_train_time, 2),
    "epochs_trained": len(history.history['accuracy'])
}

with open('/content/drive/MyDrive/Brain_Tumor/results_approach2.json', 'w') as f:
    json.dump(results_approach2, f, indent=4)

model.save('/content/drive/MyDrive/Brain_Tumor/end_to_end_efficientnet.keras')
print("Results and model saved:", results_approach2)
